In [5]:
# ==============================================================================
# 2026改修版: SVM_Radial + 外れ値解析用フル版（旧特徴量ロジック準拠）
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)
  library(stringr)
})

set.seed(42)

# -------------------------------
# パス・対象設定
# -------------------------------
base_path   <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
file_names  <- c("m_all_OH_rebuilt.csv")

# ★4目的変数を旧コードと同じように定義
target_vars_all <- c("Jsc", "Voc", "FF", "PCEmax")
# 今回このスクリプトで実際に学習するターゲット
target_vars_run <- c("PCEmax")

# 出力ルート
today    <- format(Sys.Date(), "%Y%m%d")
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
model_name <- "SVM_Radial"

# (C)(D)(E) 除外変数設定
inappropriate_vars <- c(
  "Jsccal", "JscJsccal", "PCEaTA", "PCEaTAFinal", "EQEABEST", "Rshthin", "Dresistance",
  "mobilityeblendOFET", "mobilityep1MOFET", "mobilityhblendSCLC", "mobilityhnMSCLC",
  "mobilityhp1MSCLC", "IonIoffF", "DRMS", "Var.1"
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPER 関数
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  as.numeric(r^2)
}

calc_svm_radial_importance <- function(model, data_scaled, target, features) {
  base_pred <- predict(model, data_scaled[, features, drop = FALSE])
  base_r2   <- safe_r2(data_scaled[[target]], base_pred)
  
  out_imps <- list()
  for (col in features) {
    temp <- data_scaled
    temp[[col]] <- sample(temp[[col]])
    shuffled_pred <- tryCatch(
      predict(model, temp[, features, drop = FALSE]),
      error = function(e) NA
    )
    if (all(is.na(shuffled_pred))) {
      out_imps[[col]] <- 0
    } else {
      new_r2 <- safe_r2(data_scaled[[target]], shuffled_pred)
      out_imps[[col]] <- max(0, base_r2 - new_r2)
    }
  }
  data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
}

flag_high_error <- function(df_pred, quantile_prob = 0.9) {
  thr <- stats::quantile(df_pred$AbsError, quantile_prob, na.rm = TRUE)
  df_pred$HighErrorFlag      <- df_pred$AbsError >= thr
  df_pred$HighErrorThreshold <- thr
  df_pred
}

# -------------------------------
# メインループ
# -------------------------------
summary_list    <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) {
    cat("[WARN] File not found:", filepath, "\n")
    next
  }
  
  df_raw <- tryCatch(
    read.csv(filepath, row.names = 1, check.names = FALSE),
    error = function(e) NULL
  )
  if (is.null(df_raw)) {
    cat("[WARN] Failed to read:", filepath, "\n")
    next
  }
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL
  
  for (target in target_vars_run) {
    if (!target %in% colnames(df_raw)) {
      cat("[WARN] Target not found in file:", target, "in", file, "\n")
      next
    }
    
    # 出力ディレクトリ
    tag     <- paste0(tools::file_path_sans_ext(file), "_", target)
    run_dir <- file.path(out_root, paste0(model_name, "_", tag))
    if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)
    
    # ----------------------------
    # クレンジング
    # ----------------------------
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) {
      cat("[WARN] Too few rows after na.omit:", nrow(df_temp), "for", tag, "\n")
      next
    }
    
    # ----------------------------
    # 特徴量候補（★旧ロジック：4目的変数をまとめて除外）
    # ----------------------------
    features <- setdiff(colnames(df_temp), target_vars_all)
    features <- setdiff(features, inappropriate_vars)
    if (length(physical_exclude_patterns) > 0) {
      features <- features[!grepl(paste(physical_exclude_patterns, collapse = "|"), features)]
    }
    
    # ゼロ分散除外
    vars     <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]
    
    # 多重共線性対策
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) == 0) {
      cat("[WARN] No usable features for", tag, "\n")
      next
    }
    
    # ----------------------------
    # スケーリング [-1,1]
    # ----------------------------
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1
    df_scaled$SampleID <- rownames(df_scaled)
    
    # ----------------------------
    # 10-fold CV 設定 & 学習
    # ----------------------------
    train_ctrl <- trainControl(
      method = "cv",
      number = 10,
      savePredictions = "final",
      returnResamp   = "all"
    )
    
    tune_grid <- expand.grid(
      sigma = 2^seq(-15, 3, length = 7),
      C     = c(1, 10, 100)
    )
    
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE],
        y = df_scaled[[target]],
        method    = "svmRadial",
        trControl = train_ctrl,
        tuneGrid  = tune_grid,
        metric    = "RMSE"
      ),
      error = function(e) {
        cat("  [ERROR] SVM_Radial failed:", e$message, "\n")
        return(NULL)
      }
    )
    if (is.null(model)) next
    
    # 最適パラメータをコンソール表示
    cat(sprintf(
      "[BEST] %s - %s | sigma=%.5f, C=%.3f | RMSE=%.4f, Rsq=%.4f\n",
      file, target,
      model$bestTune$sigma, model$bestTune$C,
      model$results$RMSE[model$results$sigma == model$bestTune$sigma &
                         model$results$C     == model$bestTune$C],
      model$results$Rsquared[model$results$sigma == model$bestTune$sigma &
                             model$results$C     == model$bestTune$C]
    ))
    
    # チューニング結果保存
    tune_tbl <- model$results
    tune_tbl$File   <- file
    tune_tbl$Target <- target
    tune_tbl$Model  <- model_name
    write.csv(
      tune_tbl,
      file.path(run_dir, paste0(tag, "_tuning_results.csv")),
      row.names = FALSE
    )
    
    best_row <- model$results[model$results$sigma == model$bestTune$sigma &
                                model$results$C     == model$bestTune$C, ]
    best_row$File   <- file
    best_row$Target <- target
    best_row$Model  <- model_name
    write.csv(
      best_row,
      file.path(run_dir, paste0(tag, "_best_params.csv")),
      row.names = FALSE
    )
    
    # ----------------------------
    # OOF 予測 + foldID + 誤差
    # ----------------------------
    pred_oof <- model$pred %>%
      dplyr::filter(sigma == model$bestTune$sigma,
                    C     == model$bestTune$C) %>%
      mutate(
        SampleID = rownames(df_scaled)[rowIndex],
        foldID   = Resample,
        Type     = "CV10_OOF",
        Observed = obs,
        Predicted = pred,
        Error    = Predicted - Observed,
        AbsError = abs(Error)
      ) %>%
      dplyr::select(SampleID, foldID, Observed, Predicted, Error, AbsError, Type)
    
    pred_oof <- flag_high_error(pred_oof, quantile_prob = 0.9)
    
    # 特徴量テーブル結合
    feat_df <- df_scaled[, c("SampleID", features), drop = FALSE]
    pred_with_feat <- pred_oof %>% left_join(feat_df, by = "SampleID")
    
    # モデル保存
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))
    saveRDS(pp,    file.path(run_dir, paste0(tag, "_preprocess.rds")))
    
    # CSV 出力
    write.csv(
      pred_oof,
      file.path(run_dir, paste0(tag, "_pred_OOF_withError.csv")),
      row.names = FALSE
    )
    write.csv(
      pred_with_feat,
      file.path(run_dir, paste0(tag, "_pred_OOF_withError_andFeatures.csv")),
      row.names = FALSE
    )
    write.csv(
      pred_with_feat[pred_with_feat$HighErrorFlag, ],
      file.path(run_dir, paste0(tag, "_highErrorSamples.csv")),
      row.names = FALSE
    )
    
    # 重要度
    imp_df <- calc_svm_radial_importance(model, df_scaled, target, features)
    imp_df$File   <- file
    imp_df$Target <- target
    imp_df$Model  <- model_name
    importance_list[[length(importance_list) + 1]] <- imp_df
    
    # サマリー
    cv10_r2   <- safe_r2(pred_oof$Observed, pred_oof$Predicted)
    cv10_rmse <- rmse(pred_oof$Observed, pred_oof$Predicted)
    
    summary_list[[length(summary_list) + 1]] <- data.frame(
      File        = file,
      Target      = target,
      Model       = model_name,
      CV10_R2     = cv10_r2,
      CV10_RMSE   = cv10_rmse,
      n_samples   = nrow(df_scaled),
      n_features  = length(features),
      Best_Sigma  = model$bestTune$sigma,
      Best_C      = model$bestTune$C,
      HighErrorThreshold = unique(pred_oof$HighErrorThreshold)
    )
    
    cat(sprintf("  Finished: %s - %s | CV10_R2: %.3f | n=%d | p=%d\n",
                file, target, cv10_r2, nrow(df_scaled), length(features)))
  }
}

# -------------------------------
# グローバル CSV 出力
# -------------------------------
if (length(summary_list) > 0) {
  all_summary <- do.call(rbind, summary_list)
  write.csv(
    all_summary,
    file.path(out_root, paste0("all_summary_CV10_", model_name, "_oldFeatLogic.csv")),
    row.names = FALSE
  )
}

if (length(importance_list) > 0) {
  all_imp <- do.call(rbind, importance_list)
  write.csv(
    all_imp,
    file.path(out_root, paste0("all_importance_", model_name, "_oldFeatLogic.csv")),
    row.names = FALSE
  )
}

cat("\n[DONE] SVM_Radial Outlier-aware Analysis (old feature-selection logic) completed.\n")


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."


[BEST] m_all_OH_rebuilt.csv - PCEmax | sigma=0.01562, C=10.000 | RMSE=1.2663, Rsq=0.7539
  Finished: m_all_OH_rebuilt.csv - PCEmax | CV10_R2: 0.750 | n=1045 | p=427

[DONE] SVM_Radial Outlier-aware Analysis (old feature-selection logic) completed.
